# FounderAI Colab Train + Eval V2

Notebook dedie au retrain sur Colab avec evaluation complete:
- train loss
- validation loss
- test loss
- perplexity
- signaux d'overfit
- courbe de loss
- eval comportementale problem statement
- eval multi-modules (`validation`, `ICP`, `business`, `GTM`, `market sizing`, `ROI`, `research`, `journey`)

Usage:
1. Runtime > Change runtime type
2. Choisis `T4 GPU` si disponible
3. Clique `Run all`


In [ ]:
import subprocess
from pathlib import Path

RUN_ROOT = Path('/content/founderai-colab-train-eval')
REPO_URL = 'https://github.com/leyeleye22/FounderAI.git'

if not RUN_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(RUN_ROOT)], check=True)
else:
    print('Repo already present at', RUN_ROOT)
    subprocess.run(['git', '-C', str(RUN_ROOT), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(RUN_ROOT), 'reset', '--hard', 'origin/main'], check=True)

LOG_DIR = RUN_ROOT / 'colab_outputs' / 'logs'
OUTPUT_DIR = RUN_ROOT / 'colab_outputs' / 'lora_adapter'
LOG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Repo root:', RUN_ROOT)
print('Log dir:', LOG_DIR)
print('Output dir:', OUTPUT_DIR)


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Set safer CUDA allocation mode for Colab GPUs.')


In [ ]:
!pip install -q transformers>=4.51.0 peft>=0.10.0 accelerate>=0.28.0 bitsandbytes>=0.43.0 datasets>=2.18.0 trl>=0.8.0 torch>=2.2.0 sentencepiece>=0.2.0 protobuf>=4.25.0 matplotlib>=3.8.0 pydantic>=2.12.0,<3 pydantic-settings>=2.10.1 httpx>=0.28.1 huggingface_hub>=0.34.0


In [ ]:
!nvidia-smi


In [ ]:
import os

os.environ['FOUNDER_AI_COLAB_BASE_MODEL'] = 'Qwen/Qwen3-4B'
os.environ['FOUNDER_AI_COLAB_DATA_PATH'] = str(RUN_ROOT / 'training_data' / 'teranga_merged.jsonl')
os.environ['FOUNDER_AI_COLAB_OUTPUT_DIR'] = str(OUTPUT_DIR)
os.environ['FOUNDER_AI_COLAB_METRICS_PATH'] = str(OUTPUT_DIR / 'training_metrics.json')
os.environ['FOUNDER_AI_COLAB_HISTORY_PATH'] = str(OUTPUT_DIR / 'training_history.json')
os.environ['FOUNDER_AI_COLAB_REPORT_PATH'] = str(OUTPUT_DIR / 'training_report.md')
os.environ['FOUNDER_AI_COLAB_PLOT_PATH'] = str(OUTPUT_DIR / 'loss_curve.png')
os.environ['FOUNDER_AI_COLAB_EVAL_SUMMARY_JSON'] = str(OUTPUT_DIR / 'behavioral_eval_summary.json')
os.environ['FOUNDER_AI_COLAB_EVAL_SUMMARY_MD'] = str(OUTPUT_DIR / 'behavioral_eval_summary.md')
os.environ['FOUNDER_AI_COLAB_RESET_OUTPUT'] = 'true'
os.environ['FOUNDER_AI_COLAB_USE_4BIT'] = 'true'
os.environ['FOUNDER_AI_COLAB_EPOCHS'] = '2'
os.environ['FOUNDER_AI_COLAB_MAX_SEQ_LENGTH'] = '512'
os.environ['FOUNDER_AI_COLAB_GRAD_ACCUM'] = '8'
os.environ['FOUNDER_AI_COLAB_SAVE_STEPS'] = '10'
os.environ['FOUNDER_AI_COLAB_EVAL_STEPS'] = '10'
os.environ['FOUNDER_AI_COLAB_SAMPLE_LIMIT'] = '0'
print('Training + eval config ready for a fresh full-train run.')


In [ ]:
import subprocess
from pathlib import Path

log_path = LOG_DIR / 'training_run.log'
script_path = RUN_ROOT / 'training_data' / 'train_qwen3_lora_colab.py'

with log_path.open('w', encoding='utf-8') as log_file:
    result = subprocess.run(
        ['python', str(script_path)],
        cwd=str(RUN_ROOT),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

log_text = log_path.read_text(encoding='utf-8')
print(log_text[:12000])
if result.returncode != 0:
    print('\n--- LOG TAIL ---\n')
    print(log_text[-12000:])
    raise RuntimeError(f'Training failed with exit code {result.returncode}. Full log saved at {log_path}')


In [ ]:
import json
from pathlib import Path

metrics_path = OUTPUT_DIR / 'training_metrics.json'
report_path = OUTPUT_DIR / 'training_report.md'
history_path = OUTPUT_DIR / 'training_history.json'

if not metrics_path.exists():
    raise FileNotFoundError(f'Missing metrics file at {metrics_path}')

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
summary = {
    'train_loss': metrics.get('train_loss'),
    'validation_loss': metrics.get('validation_loss'),
    'test_loss': metrics.get('test_loss'),
    'validation_perplexity': metrics.get('validation_perplexity'),
    'test_perplexity': metrics.get('test_perplexity'),
    'overfit_analysis': metrics.get('overfit_analysis'),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
from IPython.display import Image, Markdown, display

plot_path = OUTPUT_DIR / 'loss_curve.png'
report_path = OUTPUT_DIR / 'training_report.md'

if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print('No loss curve found.')

if report_path.exists():
    display(Markdown(report_path.read_text(encoding='utf-8')))
else:
    print('No markdown report found.')


In [ ]:
import subprocess
from pathlib import Path

eval_log_path = LOG_DIR / 'behavioral_eval.log'
eval_script_path = RUN_ROOT / 'training_data' / 'run_colab_full_eval.py'

with eval_log_path.open('w', encoding='utf-8') as log_file:
    result = subprocess.run(
        ['python', str(eval_script_path)],
        cwd=str(RUN_ROOT),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

log_text = eval_log_path.read_text(encoding='utf-8')
print(log_text[:12000])
if result.returncode != 0:
    print('\n--- EVAL LOG TAIL ---\n')
    print(log_text[-12000:])
    raise RuntimeError(f'Behavioral eval failed with exit code {result.returncode}. Full log saved at {eval_log_path}')


In [ ]:
import json
from IPython.display import Markdown, display

behavior_json = OUTPUT_DIR / 'behavioral_eval_summary.json'
behavior_md = OUTPUT_DIR / 'behavioral_eval_summary.md'

if behavior_json.exists():
    payload = json.loads(behavior_json.read_text(encoding='utf-8'))
    compact = {
        'problem_statement_eval': payload.get('problem_statement_eval'),
        'conversational_eval': payload.get('conversational_eval'),
        'top_failures': payload.get('top_failures', [])[:3],
    }
    print(json.dumps(compact, ensure_ascii=False, indent=2))
else:
    raise FileNotFoundError(f'Missing behavioral eval summary at {behavior_json}')

if behavior_md.exists():
    display(Markdown(behavior_md.read_text(encoding='utf-8')))
else:
    print('No behavioral eval markdown report found.')


In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(str(OUTPUT_DIR.parent / 'founderai_lora_adapter_train_eval_v2'), 'zip', str(OUTPUT_DIR))
print('Created archive:', archive_path)
files.download(archive_path)
